# QaptaanLM 3-Way Coding Benchmark: Base vs CPT vs SFT
### Head-to-Head Evaluation across HumanEval (164 tasks) & MBPP (257 tasks)

This notebook evaluates three stages of the **QaptaanLM / Qwen3.5** family on official Python code generation benchmarks:
1. **Base Foundation Model**: `Qwen/Qwen3.5-0.8B-Base` (870M parameters)
2. **CPT Model (Stage 1)**: `kaptaan45/QaptaanLM-0.75B` (752M parameters, 1B code tokens)
3. **SFT Model (Stage 2)**: `kaptaan45/QaptaanLM-0.75B-Instruct` / Kaggle SFT Checkpoint (100M instruction tokens)

**Benchmark Specs:**
- **HumanEval**: 164 hand-crafted Python programming tasks (`pass@1` greedy execution sandbox)
- **MBPP (Sanitized)**: 257 basic Python programming problems (`pass@1` test sandbox)
- **Execution Engine**: Isolated in-memory Python execution sandbox with full standard library imports.

## 1. Clean Dependencies & Setup Environment

In [ ]:
# 1. Clean dependencies (Keep native torch to preserve CUDA drivers)
!pip uninstall -y torchvision torchaudio
!pip install -q --upgrade transformers accelerate datasets safetensors tabulate sympy


## 2. Safe Imports & Hardware Initialization

In [ ]:
import gc
import json
import math
import os
import re
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Double-patch vision/torchvision to prevent C-extension issues in Python 3.12/3.10
try:
    import transformers.utils.import_utils as _iu
    _iu.is_torchvision_available = lambda *a, **kw: False
    _iu._torchvision_available = False
    _iu.is_torchvision_v2_available = lambda *a, **kw: False
    _iu._torchvision_v2_available = False
    _iu.is_vision_available = lambda *a, **kw: False
except Exception:
    pass

try:
    import transformers.image_utils as _img_u
    _img_u.is_torchvision_available = lambda *a, **kw: False
    _img_u.is_vision_available = lambda *a, **kw: False
except Exception:
    pass

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import HTML, display
from tqdm.auto import tqdm

# Model IDs / Checkpoint Paths
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

# Auto-locate SFT model (HF hub or local Kaggle input directory)
SFT_CANDIDATES = [
    "kaptaan45/QaptaanLM-0.75B-Instruct",
    "/kaggle/input/checkpoints-sft/jax_sft_hf",
    "/kaggle/input/checkpoints-sft",
    "/kaggle/input/checkpoints_sft/jax_sft_hf",
    "/kaggle/working/sft_model",
    "checkpoints/jax_sft_hf",
]
SFT_MODEL_ID = next((p for p in SFT_CANDIDATES if os.path.exists(p)), "kaptaan45/QaptaanLM-0.75B-Instruct")

# Evaluation Limits (Set to 164 / 257 for full benchmark execution)
LIMIT_HUMANEVAL = 164  # Full HumanEval benchmark (164 tasks)
LIMIT_MBPP = 257       # Full MBPP sanitized benchmark (257 tasks)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"[OK] Hardware: {DEVICE} ({NUM_GPUS} GPUs available)")
for i in range(NUM_GPUS):
    print(f"     GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"[OK] Precision: {DTYPE}")
print(f"[OK] Base Model : {BASE_MODEL_ID}")
print(f"[OK] CPT Model  : {CPT_MODEL_ID}")
print(f"[OK] SFT Model  : {SFT_MODEL_ID}")


## 3. Fast In-Memory Code Execution Sandbox & Code Extraction

In [ ]:
def run_sandbox_test(code: str, test_code: str, entry_point: Optional[str] = None) -> bool:
    """Execute Python code alongside unit tests in an isolated in-memory environment."""
    full_program = (
        "import sys, math, collections, itertools, functools, re, heapq, bisect\n"
        "from typing import Any, Dict, List, Optional, Set, Tuple, Union, Callable\n\n"
        f"{code}\n\n"
        f"{test_code}\n"
    )
    if entry_point and f"check({entry_point})" not in test_code and "check(" in test_code:
        full_program += f"\ncheck({entry_point})\n"
    
    try:
        local_ns = {}
        exec(full_program, {}, local_ns)
        return True
    except Exception:
        return False


def clean_humaneval_code(prompt: str, gen: str, is_instruct: bool = False) -> str:
    """Extract and clean Python code from model generation."""
    if is_instruct:
        # Instruct models often generate markdown code fences
        if "```python" in gen:
            code_block = gen.split("```python")[1].split("```")[0]
            return code_block.strip()
        elif "```" in gen:
            code_block = gen.split("```")[1].split("```")[0]
            return code_block.strip()

    # For base / completion style generation
    for stop_str in ["\ndef ", "\nclass ", "\nif __name__", "\nprint(", "\nassert "]:
        if stop_str in gen:
            gen = gen.split(stop_str)[0]
    if "```python" in gen:
        gen = gen.split("```python")[1].split("```")[0]
    elif "```" in gen:
        gen = gen.split("```")[1].split("```")[0]
    return prompt + gen


def extract_mbpp_code(gen: str, is_instruct: bool = False) -> str:
    """Extract executable code from MBPP generation."""
    if is_instruct:
        if "```python" in gen:
            return gen.split("```python")[1].split("```")[0].strip()
        elif "```" in gen:
            return gen.split("```")[1].split("```")[0].strip()

    for stop_str in ["\nassert ", "\nif __name__", "\nprint("]:
        if stop_str in gen:
            gen = gen.split(stop_str)[0]
    if "```python" in gen:
        return gen.split("```python")[1].split("```")[0].strip()
    if "```" in gen:
        return gen.split("```")[1].split("```")[0].strip()
    lines = [l for l in gen.split("\n") if not l.startswith("##") and not l.startswith("Explanation:")]
    return "\n".join(lines).strip()


## 4. Download Official Coding Benchmark Datasets

In [ ]:
from datasets import load_dataset

print("Downloading Official Coding Benchmark Datasets...")
DATASETS = {}

# 1. HumanEval
try:
    ds = load_dataset("openai_humaneval", split="test")
    DATASETS["HumanEval"] = list(ds)[:LIMIT_HUMANEVAL]
    print(f" [OK] HumanEval: {len(DATASETS['HumanEval'])} problems")
except Exception as e:
    print(f" [WARN] HumanEval: {e}")

# 2. MBPP (Sanitized)
try:
    ds = load_dataset("google-research-datasets/mbpp", "sanitized", split="test")
    DATASETS["MBPP"] = list(ds)[:LIMIT_MBPP]
    print(f" [OK] MBPP: {len(DATASETS['MBPP'])} problems")
except Exception as e:
    print(f" [WARN] MBPP: {e}")


## 5. Model Evaluation Pipeline

In [ ]:
def evaluate_coding_model(model_id_or_path: str, is_sft: bool = False) -> Dict[str, Dict[str, Any]]:
    """Evaluates a model across HumanEval and MBPP pass@1 metrics."""
    print("\n" + "=" * 75)
    display_name = model_id_or_path.split("/")[-1]
    model_type_label = "SFT (Instruct)" if is_sft else ("CPT Foundation" if "Qaptaan" in display_name else "Base Model")
    print(f"Loading & Evaluating: {display_name} ({model_type_label})")
    print("=" * 75)
    
    tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    stop_token_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id not in stop_token_ids:
        stop_token_ids.append(im_end_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id_or_path,
        torch_dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    model.eval()
    scores = {}

    # --- A. HumanEval (Coding pass@1) ---
    if "HumanEval" in DATASETS:
        tasks = DATASETS["HumanEval"]
        passed = 0
        desc = f"[{display_name}] HumanEval"
        
        for row in tqdm(tasks, desc=desc):
            if is_sft:
                # ChatML prompt for Instruct model
                prompt_text = f"Complete the following Python function:\n```python\n{row['prompt']}\n```"
                chat_text = f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n```python\n{row['prompt']}"
            else:
                chat_text = row["prompt"]
                
            inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    do_sample=False,
                    eos_token_id=stop_token_ids,
                    pad_token_id=tokenizer.pad_token_id,
                    repetition_penalty=1.05,
                    use_cache=True,
                )
            gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            full_code = clean_humaneval_code(row["prompt"], gen, is_instruct=is_sft)
            
            if run_sandbox_test(full_code, row["test"], row["entry_point"]):
                passed += 1
                
        score = round(passed / len(tasks) * 100.0, 2)
        scores["HumanEval"] = {
            "score": score,
            "passed": passed,
            "total": len(tasks),
            "benchmark": "HumanEval",
            "domain": "Python Code Generation",
            "metric": "pass@1",
        }
        print(f" -> HumanEval pass@1: {score}% ({passed}/{len(tasks)})")

    # --- B. MBPP (Coding pass@1) ---
    if "MBPP" in DATASETS:
        tasks = DATASETS["MBPP"]
        passed = 0
        desc = f"[{display_name}] MBPP"
        mbpp_prefix = '"""\nWrite a function to find the shared elements from two lists.\n"""\ndef similar_elements(t1, t2):\n    return tuple(set(t1) & set(t2))\n\n'
        
        for row in tqdm(tasks, desc=desc):
            if is_sft:
                prompt_text = f"Write a Python function to solve this task:\n{row['prompt']}\nProvide only executable Python code."
                chat_text = f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n```python\n"
            else:
                chat_text = f'{mbpp_prefix}"""\n{row["prompt"]}\n"""\n'
                
            inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    do_sample=False,
                    eos_token_id=stop_token_ids,
                    pad_token_id=tokenizer.pad_token_id,
                    repetition_penalty=1.05,
                    use_cache=True,
                )
            gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            code = extract_mbpp_code(gen, is_instruct=is_sft)
            test_code = (row.get("test_setup_code", "") + "\n") + "\n".join(row.get("test_list", []))
            
            if run_sandbox_test(code, test_code):
                passed += 1
                
        score = round(passed / len(tasks) * 100.0, 2)
        scores["MBPP"] = {
            "score": score,
            "passed": passed,
            "total": len(tasks),
            "benchmark": "MBPP",
            "domain": "Python Problem Solving",
            "metric": "pass@1",
        }
        print(f" -> MBPP pass@1: {score}% ({passed}/{len(tasks)})")

    # Cleanup GPU memory
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return scores


## 6. Run 3-Way Benchmark Execution

In [ ]:
print("=" * 75)
print("STARTING 3-WAY HEAD-TO-HEAD CODING BENCHMARK")
print("=" * 75)

# 1. Base Model
base_results = evaluate_coding_model(BASE_MODEL_ID, is_sft=False)

# 2. CPT Model
cpt_results = evaluate_coding_model(CPT_MODEL_ID, is_sft=False)

# 3. SFT Model
sft_results = evaluate_coding_model(SFT_MODEL_ID, is_sft=True)

print("\n✓ All evaluations completed successfully!")


## 7. Render 3-Way Leaderboard & Export Artifacts

In [ ]:
import json

benchmarks = ["HumanEval", "MBPP"]

html_scorecard = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1000px; margin: 0 auto;">
  <div style="background: linear-gradient(135deg, #1e293b, #0f172a); padding: 24px; border-radius: 12px; border: 1px solid #334155; margin-bottom: 24px; text-align: center;">
    <h2 style="color: #38bdf8; margin: 0 0 8px 0; font-size: 24px;">QaptaanLM 3-Way Coding Benchmark Leaderboard</h2>
    <p style="color: #94a3b8; margin: 0; font-size: 14px;">
      Base (<strong>{BASE_MODEL_ID}</strong>) vs CPT (<strong>{CPT_MODEL_ID}</strong>) vs SFT (<strong>{SFT_MODEL_ID.split('/')[-1]}</strong>)
    </p>
  </div>
  <table style="width: 100%; border-collapse: collapse; background: #0f172a; border-radius: 10px; overflow: hidden; border: 1px solid #334155; font-size: 14px;">
    <thead>
      <tr style="background: #1e293b; color: #94a3b8; text-transform: uppercase; font-size: 12px;">
        <th style="padding: 14px 16px; text-align: left;">Benchmark</th>
        <th style="padding: 14px 16px; text-align: center;">Metric</th>
        <th style="padding: 14px 16px; text-align: center;">Base (0.8B)</th>
        <th style="padding: 14px 16px; text-align: center;">CPT (0.75B)</th>
        <th style="padding: 14px 16px; text-align: center; color: #a855f7;">SFT (Instruct)</th>
        <th style="padding: 14px 16px; text-align: center;">SFT vs Base</th>
      </tr>
    </thead>
    <tbody>
"""

md_table = "# QaptaanLM 3-Way Coding Benchmark Leaderboard\n\n"
md_table += f"| Benchmark | Metric | {BASE_MODEL_ID.split('/')[-1]} (Base) | {CPT_MODEL_ID.split('/')[-1]} (CPT) | **SFT (Instruct)** | SFT vs Base |\n"
md_table += "| :--- | :--- | :---: | :---: | :---: | :---: |\n"

for b in benchmarks:
    b_data = base_results.get(b, {})
    c_data = cpt_results.get(b, {})
    s_data = sft_results.get(b, {})
    
    s_base = b_data.get("score", 0.0)
    s_cpt = c_data.get("score", 0.0)
    s_sft = s_data.get("score", 0.0)
    metric = s_data.get("metric", "pass@1")
    
    delta_total = s_sft - s_base
    delta_color = "#4ade80" if delta_total > 0 else ("#f87171" if delta_total < 0 else "#94a3b8")
    delta_str = f"+{delta_total:.2f}%" if delta_total > 0 else f"{delta_total:.2f}%"
    
    html_scorecard += f"""
      <tr style="border-bottom: 1px solid #1e293b;">
        <td style="padding: 12px 16px; color: #f8fafc; font-weight: bold;">{b}</td>
        <td style="padding: 12px 16px; text-align: center; color: #94a3b8;">{metric}</td>
        <td style="padding: 12px 16px; text-align: center; color: #60a5fa;">{s_base:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: #38bdf8;">{s_cpt:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: #c084fc; font-weight: bold;">{s_sft:.2f}%</td>
        <td style="padding: 12px 16px; text-align: center; color: {delta_color}; font-weight: bold;">{delta_str}</td>
      </tr>
    """
    md_table += f"| **{b}** | `{metric}` | {s_base:.2f}% | {s_cpt:.2f}% | **{s_sft:.2f}%** | `{delta_str}` |\n"

html_scorecard += """
    </tbody>
  </table>
</div>
"""

display(HTML(html_scorecard))

# Export artifacts
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "benchmark_coding_results.json", "w", encoding="utf-8") as f:
    json.dump({"base": base_results, "cpt": cpt_results, "sft": sft_results}, f, indent=2)

with open(out_dir / "benchmark_coding_report.md", "w", encoding="utf-8") as f:
    f.write(md_table)

with open(out_dir / "benchmark_coding_leaderboard.html", "w", encoding="utf-8") as f:
    f.write(html_scorecard)

print(f"\n[OK] Successfully exported artifacts to: {out_dir}")
print(" - benchmark_coding_results.json")
print(" - benchmark_coding_report.md")
print(" - benchmark_coding_leaderboard.html")
